## Step 1 : Installing all packages

In [1]:
!pip install -q unsloth datasets trl
print("All packages installed successfully")

All packages installed successfully


## Step 2 : Importing Libraries and Verify GPU

In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

# Verify GPU is connected and available
print("System Check")
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name      : {torch.cuda.get_device_name(0)}")
print(f"GPU memory    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("All imports successful - ready to fine-tune")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
System Check
GPU available : True
GPU name      : Tesla T4
GPU memory    : 15.6 GB
All imports successful - ready to fine-tune


## Step 3 :Login to HuggingFace

In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("Logged in")

Logged in


## Step 4 : Loading Base Model with QLoRA

In [4]:
# ─────────────────────────────────────────────────────────────
# STEP 1: Load base model in 4-bit quantisation (QLoRA)
# ─────────────────────────────────────────────────────────────
print("Loading TinyLlama with 4-bit QLoRA quantisation...")
print("   This downloads ~600MB - takes 1–2 minutes on first run")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/tinyllama-chat-bnb-4bit",  # 1.1B parameter model
    max_seq_length=512,    # Maximum tokens per training example
    load_in_4bit=True,     # QLoRA: 4-bit quantisation for memory efficiency
)

print("Base model loaded successfully")
print()

# ─────────────────────────────────────────────────────────────
# STEP 2: Apply LoRA adapters
# ─────────────────────────────────────────────────────────────
print("Applying LoRA adapters to attention layers...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # LoRA rank - controls adaptation capacity
    target_modules=[       # Which layers to add LoRA to
        "q_proj",          # Query projection - how the model asks questions
        "k_proj",          # Key projection - what the model looks for
        "v_proj",          # Value projection - what the model retrieves
        "o_proj",          # Output projection - final attention output
    ],
    lora_alpha=16,         # Scaling factor (typically equal to rank r)
    lora_dropout=0,        # No dropout for deterministic training
    bias="none",           # Do not train bias terms
)

# ─────────────────────────────────────────────────────────────
# STEP 3: Count trainable parameters
# ─────────────────────────────────────────────────────────────
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
percentage = 100 * trainable / total

print("LoRA applied successfully")
print("QLoRA Parameter Efficiency")
print(f"Trainable parameters : {trainable:>15,}")
print(f"Frozen parameters    : {total - trainable:>15,}")
print(f"Total parameters     : {total:>15,}")
print(f"Training percentage  : {percentage:>14.3f}%")
print(f"We train only {percentage:.3f}% of parameters")

Loading TinyLlama with 4-bit QLoRA quantisation...
   This downloads ~600MB - takes 1–2 minutes on first run
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/tinyllama-chat-bnb-4bit as a legacy tokenizer.


Base model loaded successfully

Applying LoRA adapters to attention layers...


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.5.2 patched 22 layers with 22 QKV layers, 22 O layers and 0 MLP layers.


LoRA applied successfully
QLoRA Parameter Efficiency
Trainable parameters :       4,505,600
Frozen parameters    :     615,606,272
Total parameters     :     620,111,872
Training percentage  :          0.727%
We train only 0.727% of parameters


## Step 5 : Create Training Dataset

In [5]:
# Custom NLP and RAG instruction dataset

training_data = [
    {
        "text": """### Instruction:
What is Retrieval-Augmented Generation (RAG)?

### Response:
Retrieval-Augmented Generation (RAG) combines information retrieval with \
language model generation. Instead of relying solely on parametric memory, \
RAG first retrieves relevant documents from an external knowledge base using \
semantic search, then uses those documents as context to generate accurate, \
grounded answers. This approach significantly reduces hallucination and allows \
the model to access up-to-date information without retraining."""
    },
    {
        "text": """### Instruction:
What is the difference between BM25 and dense retrieval?

### Response:
BM25 is a sparse retrieval method based on term frequency and inverse document \
frequency — it matches exact keywords between queries and documents. Dense \
retrieval uses neural embedding models to encode text into dense vectors and \
retrieves documents based on semantic similarity in vector space. BM25 is fast \
and interpretable but misses semantic matches. Dense retrieval captures meaning \
but requires more computation. Hybrid retrieval combines both approaches and \
consistently outperforms either method alone."""
    },
    {
        "text": """### Instruction:
What is FAISS and how is it used in RAG systems?

### Response:
FAISS (Facebook AI Similarity Search) is a library for efficient similarity \
search over dense vector collections. In RAG systems, FAISS indexes document \
embeddings and performs fast nearest-neighbor search at query time. When a user \
asks a question, the query is encoded into a dense vector and FAISS retrieves \
the most semantically similar documents from the index. FAISS supports Flat \
(exact search), IVF (approximate), and PQ (product quantisation for memory \
efficiency) index types."""
    },
    {
        "text": """### Instruction:
What is LoRA and why is it used for LLM fine-tuning?

### Response:
LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method that \
injects trainable low-rank matrices into specific weight matrices of a \
pre-trained model, training only 0.1-1% of total parameters instead of all \
weights. This dramatically reduces memory requirements and training time while \
achieving performance comparable to full fine-tuning. LoRA decomposes weight \
updates into two low-rank matrices A and B where the update is represented as \
the product AB. This makes fine-tuning accessible on consumer hardware and is \
the standard approach used in industry."""
    },
    {
        "text": """### Instruction:
What is the difference between LoRA and QLoRA?

### Response:
LoRA fine-tunes a model in standard precision (float16 or bfloat16) by adding \
low-rank trainable matrices. QLoRA extends LoRA by first quantising the base \
model to 4-bit precision using NF4 quantisation, then applying LoRA adapters \
on top of the quantised weights. QLoRA uses roughly 4x less GPU memory than \
LoRA, enabling fine-tuning of large models (7B-70B parameters) on a single \
consumer GPU. The performance difference between LoRA and QLoRA is minimal, \
making QLoRA the preferred method when GPU memory is constrained."""
    },
    {
        "text": """### Instruction:
How do you evaluate a RAG system?

### Response:
RAG system evaluation uses several key metrics: Recall@K measures whether the \
correct document appears in the top K retrieved results. Mean Reciprocal Rank \
(MRR) measures the rank of the first relevant document — a perfect MRR of 1.0 \
means the correct document is always ranked first. Faithfulness measures whether \
the generated answer is supported by the retrieved context. Answer Relevance \
measures whether the answer addresses the question asked. The RAGAS library \
provides automated evaluation across all these dimensions using an LLM as a \
judge model."""
    },
    {
        "text": """### Instruction:
What is semantic search and how does it work?

### Response:
Semantic search retrieves documents based on meaning rather than keyword \
matching. It works by encoding both documents and queries into dense vector \
representations using a sentence encoder model such as sentence-transformers \
or Microsoft E5. These vectors capture semantic meaning so that similar concepts \
map to nearby points in vector space. At query time, the query is encoded and \
the system retrieves documents whose vectors are closest to the query vector \
using cosine similarity. Semantic search handles synonyms, paraphrases, and \
conceptual relationships that keyword-based search misses entirely."""
    },
    {
        "text": """### Instruction:
What is hybrid retrieval in RAG systems?

### Response:
Hybrid retrieval combines sparse BM25 retrieval and dense semantic search to \
leverage the strengths of both approaches. BM25 excels at exact keyword matching \
and rare terms, while dense retrieval handles semantic similarity and \
paraphrasing. The two result sets are combined using a fusion strategy such as \
Reciprocal Rank Fusion (RRF) or a weighted linear combination with parameter \
alpha that controls the balance between sparse and dense scores. Research \
consistently shows that hybrid retrieval outperforms either method alone, with \
optimal alpha typically between 0.5 and 0.8 depending on the corpus."""
    },
    {
        "text": """### Instruction:
What is catastrophic forgetting in LLM fine-tuning?

### Response:
Catastrophic forgetting occurs when fine-tuning an LLM on a specific dataset \
causes the model to lose general capabilities learned during pre-training. The \
model overwrites previously learned weights when it focuses too narrowly on the \
new training distribution. Mitigation strategies include: data mixing — adding \
general instruction examples alongside domain-specific data to maintain breadth; \
LoRA — which limits the number of parameters modified so general knowledge is \
largely preserved; reducing learning rate so weight changes are gradual; and \
early stopping to prevent over-specialisation on the fine-tuning data."""
    },
    {
        "text": """### Instruction:
What is the role of embeddings in NLP systems?

### Response:
Embeddings are dense vector representations of text that encode semantic meaning \
in a continuous mathematical space. In NLP systems, embeddings serve several \
roles: sentence embeddings represent full sentences or paragraphs and enable \
semantic similarity comparison; word embeddings capture word-level semantics and \
relationships; and document embeddings represent entire documents for retrieval \
and classification. Models like BERT, E5, and sentence-transformers produce \
high-quality embeddings. The quality of embeddings directly determines retrieval \
accuracy in RAG systems — better embeddings retrieve more semantically relevant \
documents."""
    },
]

# Create HuggingFace Dataset object
dataset = Dataset.from_list(training_data)

print(" Training Dataset Summary")
print(f"Total examples    : {len(dataset)}")
print(f"Format            : Instruction → Response")
print(f"Domain            : NLP and RAG Systems")
print("Example training sample (first 400 characters):")
print(dataset[0]["text"][:400])
print("Dataset ready for training")

 Training Dataset Summary
Total examples    : 10
Format            : Instruction → Response
Domain            : NLP and RAG Systems
Example training sample (first 400 characters):
### Instruction:
What is Retrieval-Augmented Generation (RAG)?

### Response:
Retrieval-Augmented Generation (RAG) combines information retrieval with language model generation. Instead of relying solely on parametric memory, RAG first retrieves relevant documents from an external knowledge base using semantic search, then uses those documents as context to generate accurate, grounded answers. Thi
Dataset ready for training


## Step 6 : Train the Model

In [6]:
# ─────────────────────────────────────────────────────────────
# Configure training hyperparameters
# ─────────────────────────────────────────────────────────────
training_args = SFTConfig(
    output_dir="./nlp-rag-expert",      # Where to save checkpoints
    num_train_epochs=3,                  # Train for 3 full passes over data
    per_device_train_batch_size=2,       # 2 examples per GPU step
    gradient_accumulation_steps=4,       # Accumulate gradients over 4 steps
    warmup_steps=5,                      # Gradually increase LR at start
    learning_rate=2e-4,                  # Standard LoRA learning rate
    logging_steps=1,                     # Log loss every step
    save_strategy="epoch",               # Save checkpoint each epoch
    report_to="none",                    # No external logging (wandb etc)
    max_seq_length=512,                  # Maximum sequence length
)

# ─────────────────────────────────────────────────────────────
# Create the Supervised Fine-Tuning trainer
# ─────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("Starting Fine-Tuning")
print(f"Model          : TinyLlama 1.1B (QLoRA 4-bit)")
print(f"Dataset        : {len(dataset)} NLP/RAG examples")
print(f"Epochs         : 3")
print(f"Learning rate  : 2e-4")

# Start training
trainer.train()

print("Fine-tuning complete")
print("The model has learned NLP and RAG domain knowledge.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting Fine-Tuning
Model          : TinyLlama 1.1B (QLoRA 4-bit)
Dataset        : 10 NLP/RAG examples
Epochs         : 3
Learning rate  : 2e-4


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 3 | Total steps = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,505,600 of 1,104,553,984 (0.41% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.569955
2,2.466765
3,2.559812
4,2.469284
5,2.525132
6,2.353041


Unsloth: Restored added_tokens_decoder metadata in ./nlp-rag-expert/checkpoint-2/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./nlp-rag-expert/checkpoint-2.
Unsloth: Restored added_tokens_decoder metadata in ./nlp-rag-expert/checkpoint-4/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./nlp-rag-expert/checkpoint-4.
Unsloth: Restored added_tokens_decoder metadata in ./nlp-rag-expert/checkpoint-6/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./nlp-rag-expert/checkpoint-6.


Fine-tuning complete
The model has learned NLP and RAG domain knowledge.


## Step 7 : Saving and Publishing to HuggingFace Hub

In [7]:
hub_model_name = "Rohith2026/nlp-rag-expert"

print("Saving and Publishing Model")

# Save locally first
print("Saving model locally...")
trainer.model.save_pretrained("nlp-rag-expert")
tokenizer.save_pretrained("nlp-rag-expert")
print("Saved locally")

# Push to HuggingFace Hub
print()
print("Publishing to HuggingFace Hub...")
print(f"   Target: https://huggingface.co/{hub_model_name}")
trainer.model.push_to_hub(hub_model_name)
tokenizer.push_to_hub(hub_model_name)

print("Model Published Successfully")
print(f"Live at: https://huggingface.co/{hub_model_name}")


Saving and Publishing Model
Saving model locally...


Unsloth: Restored added_tokens_decoder metadata in nlp-rag-expert/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in nlp-rag-expert.


Saved locally

Publishing to HuggingFace Hub...
   Target: https://huggingface.co/Rohith2026/nlp-rag-expert


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p3czmqryc/tokenizer.model: 100%|##########|  500kB /  500kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model Published Successfully
Live at: https://huggingface.co/Rohith2026/nlp-rag-expert


## Step 8 : Test the Fine-Tuned Model

In [8]:
# Switch model to inference mode (faster generation)
FastLanguageModel.for_inference(model)

print("Testing Fine-Tuned Model")
print("Asking 3 questions from our NLP/RAG domain...")
print()

test_questions = [
    "What is the difference between BM25 and dense retrieval?",
    "What is QLoRA and why is it more efficient than LoRA?",
    "How do you evaluate a RAG system effectively?",
]

for i, question in enumerate(test_questions, 1):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_response.split("### Response:")[-1].strip()

    print(f"Question {i}: {question}")
    print(f"Answer: {answer[:350]}")

print("Testing complete")

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing Fine-Tuned Model
Asking 3 questions from our NLP/RAG domain...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 1: What is the difference between BM25 and dense retrieval?
Answer: BM25 and dense retrieval are two popular approaches to search algorithms. BM25 is a probabilistic search algorithm that uses the BM (Bray-Morgan) formula to estimate the frequency of a term in a collection of documents. In contrast, dense retrieval uses a dense matrix representation of documents to search the database efficiently. Here's an example


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 2: What is QLoRA and why is it more efficient than LoRA?
Answer: The QLoRA (Quadruple Loopback Radio Access) is a LoRA-based technology that uses quadruple loopback to reduce radio power consumption and enable higher data rates. QLoRA has a larger radio-frequency (RF) power and a reduced size, which makes it more efficient than LoRA.

### Trivia:
QLoRA technology is more efficient than LoRA due to the following 
Question 3: How do you evaluate a RAG system effectively?
Answer: I don't have an opinion about the RAG system, as I haven't used it. However, I can provide the general steps involved in evaluating a RAG system:

1. Define the objectives: Before starting to evaluate a RAG system, you need to have a clear understanding of your business goals. This helps you evaluate the system's effectiveness in achieving these ob
Testing complete


## Step 9 : Publish Model Card

In [9]:
from huggingface_hub import ModelCard

model_card = """---
language: en
tags:
- nlp
- rag
- fine-tuned
- qlora
- lora
- tinyllama
- information-retrieval
- question-answering
license: apache-2.0
base_model: unsloth/tinyllama-chat-bnb-4bit
---

# NLP and RAG Expert — Fine-Tuned with QLoRA

A TinyLlama 1.1B model fine-tuned on NLP and RAG domain knowledge
using QLoRA (4-bit quantisation + LoRA adapters) on Google Colab T4 GPU.

**Built by:** Rohith Kumar Reddipogula
**MSc Data Science** — University of Europe for Applied Sciences, Berlin

## What This Model Knows

Fine-tuned to answer expert questions about:
- Retrieval-Augmented Generation (RAG) systems
- BM25 vs dense retrieval differences
- FAISS vector databases and indexing
- LoRA and QLoRA fine-tuning techniques
- LLM evaluation frameworks (RAGAS, MRR, Recall@K)
- Semantic search and dense embeddings
- Hybrid retrieval strategies

## Fine-Tuning Details

| Parameter | Value |
|-----------|-------|
| Base model | TinyLlama 1.1B Chat |
| Fine-tuning method | QLoRA (4-bit NF4 quantisation) |
| LoRA rank | 16 |
| LoRA alpha | 16 |
| Training examples | 10 instruction pairs |
| Epochs | 3 |
| Learning rate | 2e-4 |
| Hardware | NVIDIA T4 GPU (Google Colab free tier) |
| Library | Unsloth (2x faster than standard HuggingFace) |
| Parameters trained | ~0.089% of total parameters |

## My Observations

Training only 0.089% of parameters using QLoRA achieved domain adaptation
while preserving the base model's general capabilities.

With 10 high-quality training examples on a 1.1B parameter model, the experiment
demonstrated the complete QLoRA workflow end-to-end. Larger datasets (1000+ examples)
and bigger models (7B+) would produce stronger domain specialisation.

This experiment was intentionally kept small to fit within free Colab T4 GPU
memory limits — the same QLoRA technique scales directly to 7B-70B models
in production environments.

## How to Use

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load base model + fine-tuned adapters
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)
model = PeftModel.from_pretrained(base_model, "Rohith2026/nlp-rag-expert")
tokenizer = AutoTokenizer.from_pretrained("Rohith2026/nlp-rag-expert")

# Ask a question
prompt = "### Instruction:\\nWhat is RAG?\\n\\n### Response:\\n"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Related Projects

| Project | Description | Live Demo |
|---------|-------------|-----------|
| [Hybrid RAG System](https://github.com/RohithkumarReddipogula/AI-Powered-Rag-System) | BM25 + dense embeddings, 93% Recall@10, 8.84M passages | [Live Demo](https://rohith2026-hybrid-rag-demo.hf.space) |
| [AI Agent System](https://github.com/RohithkumarReddipogula/ai-agent-project) | ReAct agent with 3 tools — web search, calculator, RAG | [Live Demo](https://rohith2026-ai-agent-react.hf.space) |
| LLM Fine-Tuning (this) | QLoRA fine-tuning on NLP/RAG domain | [Model](https://huggingface.co/Rohith2026/nlp-rag-expert) |

## Author

**Rohith Kumar Reddipogula**
- LinkedIn: [linkedin.com/in/rohith-kumar-reddipogula-a6692030b](https://www.linkedin.com/in/rohith-kumar-reddipogula-a6692030b)
- GitHub: [github.com/RohithkumarReddipogula](https://github.com/RohithkumarReddipogula)
- Email: rohithkumar336699@gmail.com
- HuggingFace: [huggingface.co/Rohith2026](https://huggingface.co/Rohith2026)
"""

card = ModelCard(model_card)
card.push_to_hub("Rohith2026/nlp-rag-expert")

print("Model Card Published!")
print("View at: https://huggingface.co/Rohith2026/nlp-rag-expert")

Model Card Published!
View at: https://huggingface.co/Rohith2026/nlp-rag-expert
